# Qwen2.5 LoRA / SFT stub — IndiaTaxBench JSONL

Trains (or smoke-trains) a **Qwen2.5** causal LM to map a **scenario JSON** to **old-regime** tax components matching `india_tax_capture` / `taxcalcindia` oracle rows.

- **Default model:** `Qwen/Qwen2.5-7B-Instruct` (override with `MODEL_NAME`).
- **Data:** `../india_tax_capture/data/india_tax_rows.jsonl` (one example per line).
- **Target JSON keys:** `total`, `initial_tax`, `surcharge`, `cess`.

Run on a GPU machine with `HF_TOKEN` set. Cells below use tiny `max_steps` for a **stub**; increase for real training.

In [ ]:
import os
import json
from pathlib import Path

# Uncomment in your environment:
# !pip install -q transformers peft accelerate bitsandbytes trl torch

os.environ.setdefault("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")
MODEL_NAME = os.environ["MODEL_NAME"]
REPO_ROOT = Path("..").resolve()
JSONL = REPO_ROOT / "india_tax_capture" / "data" / "india_tax_rows.jsonl"
assert JSONL.is_file(), JSONL

In [ ]:
def load_sft_rows(path: Path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        resp = obj.get("response") or {}
        if resp.get("_error"):
            continue
        old = (resp.get("tax_liability") or {}).get("old_regime") or {}
        comp = old.get("components") or {}
        scenario = (obj.get("request") or {}).get("scenario") or {}
        labels = {
            "total": old.get("total"),
            "initial_tax": comp.get("initial_tax"),
            "surcharge": comp.get("surcharge"),
            "cess": comp.get("cess"),
        }
        user = (
            "FY 2024-25 India income tax, OLD regime. Scenario JSON:\n"
            + json.dumps(scenario, ensure_ascii=False)
            + "\nRespond with JSON only: total, initial_tax, surcharge, cess (numbers)."
        )
        assistant = json.dumps(labels, ensure_ascii=False)
        rows.append({"messages": [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}]})
    return rows

examples = load_sft_rows(JSONL)
len(examples), examples[0]["messages"][0]["content"][:200]

In [ ]:
# --- LoRA attach + one forward pass (stub; expand with TRL SFTTrainer on `examples`) ---
# Requires GPU for larger Qwen; on CPU this may be slow or OOM — reduce `max_length`.

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, get_peft_model
except ImportError as e:
    raise SystemExit("Install transformers peft torch first: " + str(e))

tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token


def to_text(ex):
    parts = []
    for m in ex["messages"]:
        parts.append(f"### {m['role']}\n{m['content']}\n")
    return "\n".join(parts)


texts = [to_text(e) for e in examples[:1]]
enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=1024)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"])
model = get_peft_model(model, lora)

if torch.cuda.is_available():
    enc = {k: v.cuda() for k, v in enc.items()}
out = model(**enc, labels=enc["input_ids"])
print("stub_loss=", float(out.loss))
else:
    print("Skip forward on CPU to avoid heavy download; set CUDA_VISIBLE_DEVICES and re-run.")